In [1]:
import pandas as pd
import numpy as np
import sklearn.linear_model as skl
                
class Ridge_Regression():
    def __init__(self):
        self.parameters = np.array([])
        self.y_mean = 0
        self.X_mean = 0
    def fit(self, X, y, alpha):
        self.y_mean = np.mean(y)
        self.X_mean = np.mean(X, axis=0)
        X = X - self.X_mean
        y = y - self.y_mean
        self.parameters = np.linalg.inv(np.transpose(X) @ X + alpha * np.identity((np.transpose(X) @ X).shape[0])) @ np.transpose(X) @ y
    def predict(self, X):
        y_hat = X @ self.parameters + self.y_mean - (np.transpose(self.parameters) @ self.X_mean)
        return y_hat

def train_test_split(number_of_elements, testing_size):
    training_index = [i for i in range(0, number_of_elements)]
    testing_index = []
    for i in range(1, testing_size + 1):
        choice = np.random.choice(training_index)
        training_index.remove(choice)
        testing_index.append(choice)
    return training_index, testing_index

def kfold_cross_validation(number_of_folds, data_size):
    lists = []
    for i in range(0, number_of_folds):
        lists.append([])
    for j in range(0, data_size):
        list_index = j % number_of_folds
        lists[list_index].append(j)
    return lists

df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
X = df.drop(columns=['salary_year_avg'])
y = df['salary_year_avg'].to_numpy()
numerical_columns = ['job_posted_month_sine', 'job_posted_year']
numerical_columns = [X.columns.get_loc(x) for x in numerical_columns]
X = X.to_numpy()

training_index, testing_index = train_test_split(len(y), 5000)
X_tuning = X[testing_index]
y_tuning = y[testing_index]
X = X[training_index]
y = y[training_index]

rmses = {}
for alpha in np.logspace(-5, 2, 100):
    rmses_one_fold = []
    for item in kfold_cross_validation(5, y_tuning.shape[0]):
        for column in numerical_columns:
            X_tuning[:, column] = (X_tuning[:, column] - np.delete(X_tuning, item, axis=0)[:, column].mean()) / np.delete(X_tuning, item, axis=0)[:, column].std()
        ridge_regression = Ridge_Regression()
        ridge_regression.fit(np.delete(X_tuning, item, axis=0), np.delete(y_tuning, item), alpha)
        predictions = ridge_regression.predict(X_tuning[item])
        rmse = np.sqrt(np.mean((y_tuning[item] - predictions)**2))
        rmses_one_fold.append(rmse)
    rmses.update({alpha: np.mean(rmses_one_fold)})
best_alpha = min(rmses, key=rmses.get)
print(f'The Best Alpha Value Is {best_alpha:.4f}.')

r2_values = []
r2_values_skl = []
for item in kfold_cross_validation(5, y.shape[0]):
    for column in numerical_columns:
        X[:, column] = (X[:, column] - np.delete(X, item, axis=0)[column].mean()) / np.delete(X, item, axis=0)[column].std()
    ridge_regression = Ridge_Regression()
    ridge_regression.fit(np.delete(X, item, axis=0), np.delete(y, item), best_alpha)
    predictions = ridge_regression.predict(X[item])
    r2 = 1 - (np.sum((y[item] - predictions)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values.append(r2)
    skl_ridge_regression = skl.RidgeCV()
    model = skl_ridge_regression.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions_skl = model.predict(X[item])
    r2_skl = 1 - (np.sum((y[item] - predictions_skl)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values_skl.append(r2_skl)
print(f'The Average R Squared Score For Ridge Regression Is {np.mean(r2_values):.4f}.')
print(f'The Average R Squared Score For Scikit-Learn\'s Ridge Regression Is {np.mean(r2_values_skl):.4f}.')


The Best Alpha Value Is 12.0450.
The Average R Squared Score For Ridge Regression Is 0.3113.
The Average R Squared Score For Scikit-Learn's Ridge Regression Is 0.3118.
